# Getting started

A trellis (small multiples, facets) draws the same plot once per group of a table, on
scales that are the same everywhere, so the panels can be compared at a glance. With
anytrellis you say which columns split the table into panels (`wrap=`, or `row=` and
`col=`) and which layers each panel draws (`line`, `points`, `bars`, `rule`). Every panel
is drawn on the GPU in one canvas, so millions of marks stay smooth.

**Using a plot:** click it, then scroll to zoom along x (Shift+scroll: y; over an axis:
that axis), drag to pan, Shift+drag to zoom to a box, `u` to undo and `r` to reset. Panels
that share a scale zoom and pan together. `y` keeps y fitted to the data in view, and `s`
makes drag a brush that selects (Esc clears). Hover a mark for its values. Double-click a
panel to see it alone (Page Up and Page Down: the next ones; Backspace: all again), and
press `f` (or the toolbar's button) for full screen.

In [ ]:
from anytrellis import Trellis
from anytrellis.datasets import load

In [ ]:
cars = load("cars")
cars.head()

## One panel per group: `wrap=`

`wrap=` makes a panel per level of a column and wraps them into rows (`cols=` sets how
many columns). A colour column with numbers is drawn along a colormap.

In [ ]:
by_origin = Trellis(cars, wrap="origin", title="Fuel use and weight, by origin")
by_origin.points(x="weight", y="mpg", color="cylinders", tooltip=["name", "year"])

## Rows and columns: `row=` and `col=`

A grid crosses two columns: a row per level of one and a column per level of the other.
Every cell is kept, also those without data (there were no 3- or 5-cylinder cars from the
USA), so the rows and columns stay aligned.

In [ ]:
grid = Trellis(cars, row="origin", col="cylinders", panel_height=130)
grid.points(x="horsepower", y="mpg", size=4, tooltip=["name"])

## Free scales

Shared scales make panels comparable; free scales show each panel's own detail.
`share_y=False` gives each panel its own y scale (`"row"` or `"col"`: one per row or
column of a grid). Zooming x still moves every panel: x is shared.

In [ ]:
stocks = load("stocks")
free = Trellis(stocks, row="symbol", share_y=False, panel_height=90, title="Monthly prices")
free.line(x="date", y="price")

## Layers, context and reference lines

Each layer is drawn in every panel. `facet="exclude"` draws the *other* panels' rows in
grey behind each panel, so each stock is seen against the rest; `rule` draws a line at
fixed positions (here the day Lehman Brothers fell).

In [ ]:
layered = (
    Trellis(stocks, wrap="symbol", cols=3, title="Each stock against the others")
    .line(x="date", y="price", group="symbol", facet="exclude", width=1, name="others")
    .line(x="date", y="price", color="#1f77b4", width=2, name="price")
    .rule(x="2008-09-15", dash=[4, 3], color="#888888")
)
layered

## Order

Panels, bands and colours are in alphabetical order unless told otherwise: `order()`
takes the levels in order, or ranks them by a summary of a column (`by="median"`,
`"mean"`, `"min"`, `"max"`, `"sum"` or `"count"`).

In [ ]:
ranked = (
    Trellis(cars, wrap="origin", cols=3, title="Origins by median fuel use")
    .order("origin", by="median", of="mpg", descending=True)
    .points(x="weight", y="mpg", size=4)
)
ranked

## One panel at a time

Double-click a panel to see it alone on the whole plot, and step through the others with
Page Up and Page Down at the same zoom. From Python, `open()` does the same (by the
panel's name, see `panels()`), `close()` goes back, and `focused` says which panel the
plot shows, also when it was opened in the browser.

In [ ]:
ranked.panels()

In [ ]:
ranked.open("Japan")
ranked.focused

## Sharing a plot

`save_html` writes a page with the data and the code inside it: it opens in any browser,
without Python, and keeps zooming and hovering.

In [ ]:
layered.save_html("stocks.html")